# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

This dataset contains tabular records for 77 cancer survivors with second primary colorectal cancer, including clinicopathological and molecular variables.

Explore the dataset step by step below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata.to_json()

print("Dataset Title:")
print(metadata['name'])
print("\nDataset Description:")
print(metadata['description'])
print("\nDataset Identifier:")
print(metadata['identifier'])
print("\nAuthors:")
for author in metadata.get('author', []):
    print(f" - {author['@id']}")
print("\nRecord Sets:")
for record_set in metadata.get('recordSet', []):
    print(f" - {record_set.get('@id', record_set)}")
print("\nAvailable keywords:")
print(metadata.get('keywords', []))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

We'll list each record set and their fields/columns based on Croissant metadata. All references are made using the entity `@id`.

In [ ]:
# Print a summary of record sets, their fields, and columns by @id
croissant_record_sets = metadata.get('recordSet', [])
if not croissant_record_sets:
    print('No record sets listed directly in metadata. Attempting to use mlcroissant Dataset API to enumerate available record sets.')
    record_sets = dataset.record_sets()
else:
    record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in croissant_record_sets]

record_set_fields = {}

for rs_id in record_sets:
    print(f"\nRecord Set @id: {rs_id}")
    rs_obj = dataset.record_set(rs_id)
    if hasattr(rs_obj, 'fields'):
        try:
            fields = list(rs_obj.fields)
        except Exception:
            fields = []
    else:
        fields = []
    print("  Fields @id:")
    for fld in fields:
        if isinstance(fld, dict) and '@id' in fld:
            print(f"    - {fld['@id']}")
        elif hasattr(fld, 'id'):
            print(f"    - {fld.id}")
        else:
            print(f"    - {fld}")
    record_set_fields[rs_id] = [fld['@id'] if isinstance(fld, dict) and '@id' in fld else getattr(fld, 'id', str(fld)) for fld in fields]

    # Try to enumerate columns
    if hasattr(rs_obj, 'columns'):
        print("  Columns @id:")
        try:
            columns = list(rs_obj.columns)
        except Exception:
            columns = []
        for col in columns:
            if isinstance(col, dict) and '@id' in col:
                print(f"    - {col['@id']}")
            elif hasattr(col, 'id'):
                print(f"    - {col.id}")
            else:
                print(f"    - {col}")
    else:
        print("  No columns attribute found.")

## 3. Data Extraction
Load records from a specific record set into pandas DataFrames for analysis.

By referencing their `@id`, we extract each record set, then inspect their columns for exploration.

In [ ]:
# Gather record sets @id for extraction
record_sets = dataset.record_sets()

dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"No records found for record set {rs_id}")
        continue
    print(f"\nLoaded {len(records)} records from record set {rs_id}")
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print("Columns:", df.columns.tolist())
    print(df.head())

# For demonstration, pick the first record set to proceed
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    selected_df = dataframes[selected_record_set_id]
    print(f"\nUsing record set {selected_record_set_id} for EDA.")
else:
    selected_record_set_id = None
    selected_df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All selected fields are referenced by their `@id`.

This step demonstrates outlier removal, normalization, and grouping by a categorical field.

In [ ]:
import numpy as np

if selected_df is not None:
    # Try to infer a numeric field by scanning DataFrame columns
    numeric_fields = [col for col in selected_df.columns if np.issubdtype(selected_df[col].dtype, np.number)]
    print("Numeric fields detected:", numeric_fields)
    if len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]
        threshold = selected_df[numeric_field_id].quantile(0.1)  # Use 10th percentile as example threshold
        filtered_df = selected_df[selected_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize numeric field
        filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

        # Try to group by a categorical field
        categorical_fields = [col for col in selected_df.columns if selected_df[col].dtype == object and col != numeric_field_id]
        if len(categorical_fields) > 0:
            group_field_id = categorical_fields[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
                print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields available for filtering and normalization.")
else:
    print("No DataFrame selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the selected record set using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_df is not None and len(selected_df.columns) > 0:
    # Display histogram of numeric field
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.histplot(selected_df[numeric_field_id].dropna(), bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # If group_field identified, plot mean by group
        if 'group_field_id' in locals():
            group_means = selected_df.groupby(group_field_id)[numeric_field_id].mean()
            group_means.plot(kind='bar', figsize=(8,4))
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()
else:
    print("No DataFrame or fields available for visualization.")

## 6. Conclusion
This notebook demonstrates loading, overview, extraction, and analysis of a FAIR^2 dataset via Croissant schema using the `mlcroissant` library. Each entity was referenced by its `@id`, ensuring precise exploration and manipulation.

Key findings are accessible via summary statistics and visualizations; you may extend this workflow for deeper modeling, predictions, or validation tasks.
